# Introduction

This file shows how we set up our MySQL cloud database on [Railway](https://docs.railway.com/databases).       

The database is used in our [BikeCast app](https://bikecast-app.streamlit.app/) for visualizations of historical data and for fetching weather forecasts for the prediction of bike demand.

# Database creation

In [ ]:
#Loading the necessary packages:
import pandas as pd
import streamlit as st

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## Connecting to empty database

In [3]:
def get_db_password():
    db_password = st.secrets["RAILWAY_PASSWORD"]
    return db_password

In [4]:
def create_connection_string(db_password):

    schema = st.secrets["RAILWAY_DATABASE"]
    host = st.secrets["RAILWAY_HOST"]
    user = st.secrets["RAILWAY_USER"]
    port = int(st.secrets["RAILWAY_PORT"])

    connection_string = URL.create(
        drivername="mysql+pymysql",
        username=user,
        password=db_password,
        host=host,
        port=port,
        database=schema,
        query={"charset": "utf8mb4"}
    )

    return connection_string

In [5]:
db_password = get_db_password()

connection_string = create_connection_string(db_password)

engine = create_engine(
    connection_string,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 15}
)

In [6]:
with engine.connect() as connection:

    result = connection.execute(
        text("SELECT DATABASE();")
    )

    print("Connected to database:", result.scalar())

Connected to database: railway


## Creating empty tables in database

In [ ]:
create_cities = """CREATE TABLE IF NOT EXISTS cities (
    id INT PRIMARY KEY,
    name VARCHAR(255),
    lat DECIMAL(8,6), 
    lon DECIMAL(9,6),
    timezone VARCHAR(100),
    country CHAR(2),
    return_to_official_only BOOLEAN
)CHARACTER SET utf8mb4;
"""

*Note*: The data types for latitude and longitude were chosed based on [this discussion on StackOverflow](#https://stackoverflow.com/questions/1196415/what-datatype-to-use-when-storing-latitude-and-longitude-data-in-sql-databases) 

In [9]:
with engine.begin() as connection:
    connection.execute(text(create_cities))

print("Cities table created successfully!")

Cities table created successfully!


In [11]:
create_dataset = """
CREATE TABLE IF NOT EXISTS dataset_final (
    id INT AUTO_INCREMENT PRIMARY KEY,

    city_name VARCHAR(255),
    year SMALLINT,
    month TINYINT,
    day TINYINT,
    hour TINYINT,
    total_bike_usage INT,

    time_start_berlin_tz VARCHAR(35),
    timezone INT,

    temp DOUBLE,
    humidity INT,
    wind_speed DOUBLE,
    rain_1h DOUBLE,
    snow_1h DOUBLE,
    clouds_all INT,

    weather_main VARCHAR(100),
    weather_description VARCHAR(255),

    day_type VARCHAR(30),
    school_break TINYINT,
    uni_break TINYINT,
    working_day TINYINT,
    weather_group VARCHAR(100),

    INDEX idx_city (city_name),
    INDEX idx_year_month (year, month)
) CHARACTER SET utf8mb4;
"""

In [12]:
with engine.begin() as connection:
    connection.execute(text(create_dataset))

print("Dataset_final table created successfully!")

Dataset_final table created successfully!


## Inserting csv files into database tables

In [13]:
cities_df = pd.read_csv("cities.csv")

dataset_df = pd.read_csv(
    "dataset_final.csv",
    dtype={"time_start_berlin_tz": "string"}
)

In [14]:
display(cities_df.head())
display(dataset_df.head())

,id,name,lat,lon,timezone,country,return_to_official_only
0,220,Zagreb,45.7984,15.98810,Europe/Berlin,HR,True
1,431,Kalisz,51.7525,18.03410,Europe/Warsaw,PL,True
2,199,Innsbruck,47.2632,11.39610,Europe/Berlin,AT,True
3,519,Siemianowice Śląskie,50.3031,19.02220,Europe/Warsaw,PL,True
4,21,Karlsruhe,49.0102,8.41827,Europe/Berlin,DE,False


,city_name,year,month,day,hour,total_bike_usage,time_start_berlin_tz,timezone,temp,humidity,...,rain_1h,snow_1h,clouds_all,weather_main,weather_description,day_type,school_break,uni_break,working_day,weather_group
0,Berlin,2022,8,26,16,15,2022-08-26 16:00:00+02:00,7200.0,24.91,73,...,2.50,0.0,100,Rain,moderate rain,Weekday,0,1,1,Rain
1,Berlin,2022,8,26,17,40,2022-08-26 17:00:00+02:00,7200.0,22.12,84,...,5.43,0.0,100,Rain,heavy intensity rain,Weekday,0,1,1,Rain
2,Berlin,2022,8,26,18,67,2022-08-26 18:00:00+02:00,7200.0,20.75,91,...,3.49,0.0,100,Rain,moderate rain,Weekday,0,1,1,Rain
3,Berlin,2022,8,26,19,101,2022-08-26 19:00:00+02:00,7200.0,20.95,84,...,0.25,0.0,99,Rain,light rain,Weekday,0,1,1,Rain
4,Berlin,2022,8,26,20,107,2022-08-26 20:00:00+02:00,7200.0,20.56,87,...,0.51,0.0,99,Rain,light rain,Weekday,0,1,1,Rain


In [17]:
cities_df["return_to_official_only"] = (
    cities_df["return_to_official_only"].astype(int)
)

In [ ]:
# Checking the columns in the data frames that we just imported
print(cities_df.columns.tolist())
print(dataset_df.columns.tolist())

['id', 'name', 'lat', 'lon', 'timezone', 'country', 'return_to_official_only']
['city_name', 'year', 'month', 'day', 'hour', 'total_bike_usage', 'time_start_berlin_tz', 'timezone', 'temp', 'humidity', 'wind_speed', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'weather_description', 'day_type', 'school_break', 'uni_break', 'working_day', 'weather_group']


In [ ]:
# Checking if the columns in the pre-set empty tables match the column names from the dfs that we will insert:
expected_cities_columns = [
    "id", "name", "lat", "lon", "timezone",
    "country", "return_to_official_only"
]

expected_dataset_columns = [
    "city_name", "year", "month", "day", "hour",
    "total_bike_usage", "time_start_berlin_tz",
    "timezone", "temp", "humidity", "wind_speed",
    "rain_1h", "snow_1h", "clouds_all",
    "weather_main", "weather_description",
    "day_type", "school_break", "uni_break",
    "working_day", "weather_group"
]

assert set(cities_df.columns) == set(expected_cities_columns)
assert set(dataset_df.columns) == set(expected_dataset_columns)

print("All columns match!")

All columns match!


In [20]:
def send_df_to_sql(connection_string, df_name, table_name_in_db):

    df_name.to_sql(
        table_name_in_db,
        if_exists="append",
        con=connection_string,
        index=False,
        chunksize=500
    )

    print(f"Uploaded {len(df_name)} rows to {table_name_in_db}!")

In [21]:
with engine.connect() as connection:

    n_cities = connection.execute(
        text("SELECT COUNT(*) FROM cities")
    ).scalar()

    n_dataset = connection.execute(
        text("SELECT COUNT(*) FROM dataset_final")
    ).scalar()

print("Cities:", n_cities)
print("Dataset:", n_dataset)

Cities: 0
Dataset: 0


In [22]:
if n_cities != 0 or n_dataset != 0:
    raise ValueError(
        "One or both Railway tables already contain data. "
        "Check their contents before uploading again."
    )

In [ ]:
send_df_to_sql(
    engine,
    cities_df,
    "cities"
)

Uploaded 267 rows to cities!
The cities table has been sent to the Railway database


In [24]:
send_df_to_sql(
    engine,
    dataset_df,
    "dataset_final"
)

Uploaded 22144 rows to dataset_final!


In [ ]:
#Checking that the all the rows in both tables have been uploaded correctly
with engine.connect() as connection:

    railway_cities_count = connection.execute(
        text("SELECT COUNT(*) FROM cities")
    ).scalar()

    railway_dataset_count = connection.execute(
        text("SELECT COUNT(*) FROM dataset_final")
    ).scalar()

print("Cities:")
print("CSV:", len(cities_df))
print("Railway:", railway_cities_count)

print("\nFinal dataset:")
print("CSV:", len(dataset_df))
print("Railway:", railway_dataset_count)

Cities:
CSV: 267
Railway: 267

Final dataset:
CSV: 22144
Railway: 22144


# Database exploration

In [ ]:
#Checking if everything works with some queries
query = """
SELECT *
FROM dataset_final
LIMIT 10;
"""

test_df = pd.read_sql(query, engine)

display(test_df)

,id,city_name,year,month,day,hour,total_bike_usage,time_start_berlin_tz,timezone,temp,...,rain_1h,snow_1h,clouds_all,weather_main,weather_description,day_type,school_break,uni_break,working_day,weather_group
0,1,Berlin,2022,8,26,16,15,2022-08-26 16:00:00+02:00,7200,24.91,...,2.50,0.0,100,Rain,moderate rain,Weekday,0,1,1,Rain
1,2,Berlin,2022,8,26,17,40,2022-08-26 17:00:00+02:00,7200,22.12,...,5.43,0.0,100,Rain,heavy intensity rain,Weekday,0,1,1,Rain
2,3,Berlin,2022,8,26,18,67,2022-08-26 18:00:00+02:00,7200,20.75,...,3.49,0.0,100,Rain,moderate rain,Weekday,0,1,1,Rain
3,4,Berlin,2022,8,26,19,101,2022-08-26 19:00:00+02:00,7200,20.95,...,0.25,0.0,99,Rain,light rain,Weekday,0,1,1,Rain
4,5,Berlin,2022,8,26,20,107,2022-08-26 20:00:00+02:00,7200,20.56,...,0.51,0.0,99,Rain,light rain,Weekday,0,1,1,Rain
5,6,Berlin,2022,8,26,21,91,2022-08-26 21:00:00+02:00,7200,20.80,...,1.35,0.0,99,Rain,moderate rain,Weekday,0,1,1,Rain
6,7,Berlin,2022,8,26,22,99,2022-08-26 22:00:00+02:00,7200,20.02,...,0.13,0.0,98,Rain,light rain,Weekday,0,1,1,Rain
7,8,Berlin,2022,8,26,23,71,2022-08-26 23:00:00+02:00,7200,19.79,...,0.00,0.0,100,Clouds,overcast clouds,Weekday,0,1,1,Clouds
8,9,Berlin,2022,8,27,0,80,2022-08-27 00:00:00+02:00,7200,19.99,...,0.25,0.0,100,Rain,light rain,Weekend,0,1,0,Rain
9,10,Berlin,2022,8,27,1,76,2022-08-27 01:00:00+02:00,7200,19.99,...,0.84,0.0,99,Rain,light rain,Weekend,0,1,0,Rain


In [27]:
query = """
SELECT *
FROM cities
WHERE country = 'DE';
"""

german_cities = pd.read_sql(query, engine)

display(german_cities)

,id,name,lat,lon,timezone,country,return_to_official_only
0,1,Leipzig,51.3435,12.36370,Europe/Berlin,DE,1
1,7,Wiesbaden,50.0739,8.24181,Europe/Berlin,DE,0
2,8,Frankfurt,50.1219,8.66890,Europe/Berlin,DE,1
3,14,Köln,50.9429,6.95649,Europe/Berlin,DE,0
4,16,Bielefeld,52.0257,8.53286,Europe/Berlin,DE,1
...,...,...,...,...,...,...,...
86,810,Hückelhoven,51.0529,6.21933,Europe/Berlin,DE,1
87,811,Wegberg,51.1427,6.28130,Europe/Berlin,DE,1
88,817,Mobic,50.8493,6.66870,Europe/Berlin,DE,1
89,819,Grünheide (Mark),52.4247,13.82030,Europe/Berlin,DE,0


In [28]:
engine.dispose()